# 前一交易日尾盘有符号 SOXL 相对 SOXX 杠杆残差：来源与去重

一次原生Qlib一分钟读取、两项封存Recorder读取；只验证当日收盘后的计算和下一09:29可用性假设、覆盖与原输入去重。研究动机来自杠杆ETF再平衡价格影响论文，但此残差不是实际申赎或再平衡流。

In [1]:
import os
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
import hashlib,json
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
root=Path.cwd();card=root/'research/stage1_late_leveraged_tracking_residual_source_20261024'
plan_path=card/'plan.json';plan=json.loads(plan_path.read_text());s=plan['sources']
repair=json.loads((card/'source_group_missing_repair.json').read_text())
assert repair['status']=='mechanical_source_group_cardinality_repair_before_scoring' and repair['cumulative_budget']['Qlib_1min_feature_reads_max']==2
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and all(plan['budget'][k]==0 for k in ('new_market_GETs','new_model_fits','new_native_accounts','future_market_outcome_reads','new_Recorders'))
a=root/s['minute_card'];assert sha(a/'summary.json')==s['minute_summary_sha256']
assert sha(a/'nas_readback.json')==s['minute_NAS_readback_sha256']
assert sha(root/s['core_1min_provider']/'qlib_store_manifest.json')==s['core_1min_manifest_sha256']
sa=json.loads((a/'summary.json').read_text());rb=json.loads((a/'nas_readback.json').read_text())
assert sa['nas_evidence']['sha256']==rb['NAS_archive_sha256']==s['minute_NAS_sha256'] and rb['artifact_hashes_checked']>=118
art=a/'qlib_recorder'/s['minute_experiment_id']/s['minute_recorder_id']/'artifacts'
assert sha(art/'inputs/minute_features')==s['minute_features_sha256'] and sha(art/'inputs/features')==s['original39_features_sha256']
qlib.init(provider_uri=str((root/s['core_1min_provider']).resolve()),region='us',expression_cache=None,dataset_cache=None,kernels=1)
with R.uri_context(str((a/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['minute_experiment_id'],recorder_id=s['minute_recorder_id'])
 old=rec.load_object('inputs/minute_features');base=rec.load_object('inputs/features')
assert len(old)==2008 and len(base)==3966
assert old.index.is_unique and base.index.is_unique
bars=D.features(['SOXL','SOXX'],['$open','$close','$price_valid'],start_time='2018-10-15',end_time='2026-10-09',freq='1min')
assert set(bars.index.names)=={'datetime','instrument'}
bars=bars.reset_index();bars['datetime']=pd.to_datetime(bars.datetime);bars['date']=bars.datetime.dt.normalize()
assert set(bars.instrument.str.upper().unique())=={'SOXL','SOXX'}
avail=pd.to_datetime(old.minute_feature_available_at,utc=True).dt.tz_convert('America/New_York').dt.tz_localize(None)
close_by_day=(avail-pd.Timedelta(minutes=1));close_by_day.index=pd.DatetimeIndex(old.index).normalize()
records=[]
for (day,symbol),g in bars.groupby(['date','instrument'],sort=True):
 day=pd.Timestamp(day).normalize();symbol=str(symbol).upper()
 if day not in close_by_day.index:continue
 closing=pd.Timestamp(close_by_day.loc[day])
 if pd.isna(closing):
  records.append({'date':day,'symbol':symbol,'valid':False,'late_return30':None,'bars':0});continue
 late=g[(g.datetime>closing-pd.Timedelta(minutes=30))&(g.datetime<=closing)].sort_values('datetime')
 expected=pd.date_range(closing-pd.Timedelta(minutes=29),closing,freq='min')
 valid=bool(len(late)==30 and pd.DatetimeIndex(late.datetime).equals(expected))
 if valid:
  values=late[['$open','$close','$price_valid']].to_numpy(float)
  valid=bool(np.isfinite(values).all() and (values[:,:2]>0).all() and (values[:,2]>0).all())
 ret=float(np.log(late['$close'].to_numpy(float)/late['$open'].to_numpy(float)).sum()) if valid else None
 records.append({'date':day,'symbol':symbol,'valid':valid,'late_return30':ret,'bars':len(late)})
source=pd.DataFrame(records)
assert len(source)<=2*len(old) and source.groupby('date').symbol.nunique().le(2).all()
wide=source.pivot(index='date',columns='symbol',values='late_return30').reindex(old.index)
known=wide.SOXL.notna()&wide.SOXX.notna()&old.minute_core_ready.astype(bool)
coord=(wide.SOXL-3*wide.SOXX).where(known)
assert coord.notna().sum()>1900
train=coord.loc['2018-10-15':'2022-12-30'];held=coord.loc['2023-01-03':'2026-10-09'];held944=held.iloc[:944]
assert len(train)==1061 and len(held)>=944
coverage=[]
for year,g in coord.groupby(coord.index.year):
 if 2018<=year<=2026:coverage.append({'year':int(year),'sessions':len(g),'known':int(g.notna().sum()),'missing':int(g.isna().sum()),'partial_endpoint':year in (2018,2026)})
controls=list(sa['input_audit']['original_features'])+['minute_soxx_late_return30','minute_soxx_late_volume_share','minute_soxl_tracking_residual_rv5','minute_soxl_soxx_sign_agreement']
assert len(controls)==43 and set(controls[-4:]).issubset(old.columns)
control_frame=pd.concat([base[controls[:39]],old[controls[-4:]]],axis=1).reindex(train.index)
correlations=[]
for name in controls:
 z=pd.DataFrame({'candidate':train,'control':control_frame[name]}).dropna()
 if len(z)>=100:
  correlations.append({'control':name,'rows':len(z),'pearson':float(z.candidate.corr(z.control,method='pearson')),
                       'spearman':float(z.candidate.corr(z.control,method='spearman'))})
assert len(correlations)==43 and all(np.isfinite([x['pearson'],x['spearman']]).all() for x in correlations)
max_abs_spearman=max(abs(x['spearman']) for x in correlations);max_abs_pearson=max(abs(x['pearson']) for x in correlations)
pass_gate=bool(train.notna().sum()>=1055 and held944.notna().sum()>=940 and max_abs_spearman<.85 and max_abs_pearson<1-1e-12)
values=coord.dropna().sort_index();digest=hashlib.sha256(values.to_csv(index=True,float_format='%.17g').encode()).hexdigest()
result={'status':'conditional_source_novelty_pass_first_receipt_unqualified' if pass_gate else 'source_or_novelty_gate_failed_no_fit',
        'plan_sha256':sha(plan_path),'minute_recorder_id':s['minute_recorder_id'],'source_sessions':len(old),'native_1min_rows':len(bars),'source_symbol_day_groups':len(source),'Qlib_1min_feature_reads_total':2,
        'known_sessions':int(coord.notna().sum()),'train_sessions':len(train),'train_known':int(train.notna().sum()),
        'held_sessions':len(held),'held_known':int(held.notna().sum()),'f696_aligned_held_944_known':int(held944.notna().sum()),
        'coverage':coverage,'correlations':correlations,'max_abs_train_spearman':max_abs_spearman,'max_abs_train_pearson':max_abs_pearson,
        'candidate_min_max':[float(values.min()),float(values.max())],'derived_coordinate_csv_sha256':digest,
        'source_novelty_pass':pass_gate,'new_market_GETs':0,'new_model_fits':0,'new_native_accounts':0,'future_market_outcome_reads':0,'new_Recorders':0,
        'historical_first_receipt_certified':False,'independent_validation':False,'SOXX_goal_achieved':False,
        'limits':['Late signed ETF price residual is not measured leveraged-ETF rebalancing flow','Historical minute bars have no certified first-live receipt vintages','Coverage and low input correlation do not prove predictive information or fee-net SOXX value']}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print({'status':result['status'],'train_known':result['train_known'],'held944_known':result['f696_aligned_held_944_known'],
       'max_abs_spearman':max_abs_spearman,'top_controls':sorted(correlations,key=lambda x:abs(x['spearman']),reverse=True)[:5]})


{'status': 'conditional_source_novelty_pass_first_receipt_unqualified', 'train_known': 1060, 'held944_known': 944, 'max_abs_spearman': 0.16781195036898527, 'top_controls': [{'control': 'tracking_error_5_z20', 'rows': 1060, 'pearson': 0.09624045358778897, 'spearman': 0.16781195036898527}, {'control': 'tracking_error_5_pctile60', 'rows': 1060, 'pearson': 0.10630992758343541, 'spearman': 0.14990094624929953}, {'control': 'minute_soxx_late_return30', 'rows': 1060, 'pearson': 0.29176582517323246, 'spearman': 0.13316676279476392}, {'control': 'soxx_ret3', 'rows': 1060, 'pearson': -0.005553087603518527, 'spearman': 0.07326869300228928}, {'control': 'soxl_ret3', 'rows': 1060, 'pearson': -0.014690371797521968, 'spearman': 0.0727061307895936}]}


## 中文结论

# 前日尾盘有符号杠杆残差：条件性来源与数值去重通过

固定定义为当日收盘前30根已完成 Qlib 一分钟 raw NONE bar 的 `Σlog(SOXL close/open)−3×Σlog(SOXX close/open)`，只供**下一交易日09:29**的 f696 配置器考虑。实际早收盘按封存日程的收盘时间后推30分钟；缺 bar、无效价或源日期保持未知，不填零。杠杆ETF再平衡流可能影响尾盘价格并在次日回转的研究动机见[原研究论文](https://abarbon.com/assets/Liquidity_Provision_to_Rebalancing_Flows_from_Leveraged_ETFs_and_Equity_Options.pdf)；本式是ETF价格残差，**没有测量基金申赎或再平衡流**。

从封存原生Qlib一分钟来源读入 **1,558,560** 条双ETF记录，原分钟卡2008个日期中 **2005个**有双标的有效尾盘坐标；2018–2022训练段 **1060/1061**，2023–2026复用来源段 **945/946**，精确f696前944个信号日 **944/944**。缺值在2018、2020、2026各1日；原39项与四个已有分钟坐标的训练期最大绝对Spearman **0.16781**、最大绝对Pearson **0.29177**，最高相关是旧跟踪误差坐标，非单列复制。43项完整对照见 `result.json`。来源卡Recorder及NAS回读、core_1min Qlib manifest SHA已验证；零未来标签、拟合、账户或外部GET。

初次实现错误要求每个2008日期都存在两个Qlib分钟组，而封存来源本就允许缺失日；失败发生在未来标签与拟合之前。`attempt1_preflight_failure.json`与`source_group_missing_repair.json`保留第一遍本地Qlib读取及第二遍机械修复预算；公式、年份和去重门没有改变。

决定 `conditional_source_novelty_pass_first_receipt_unqualified`。当前历史分钟bar没有逐日首次直播接收或最终版本认证，低相关也不证明预测力。下一步只能另冻一个与09:29→09:47执行钟匹配的单期经济标签信息门；通过后仍需精确f696全链0bp Tiered自身资金账户、2/5bp和独立前向。SOXX稳定净超额尚未证明。
